In [1]:
from typing import TypedDict
from dotenv import load_dotenv

from langgraph.graph import StateGraph, START, END

from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.messages import HumanMessage

load_dotenv()

True

In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

load_dotenv()
 

class HuggingFaceLLM:

    def __init__(
        self,
        model_name="Qwen/Qwen2.5-7B-Instruct",
        api_key=None,
    ):

        self.model_name = model_name

        self.client = InferenceClient(
            api_key=api_key or os.getenv("HF_TOKEN")
        )

    def invoke(self, prompt: str) -> str:

        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        return response.choices[0].message.content
    
llm = HuggingFaceLLM()

e:\Pradhumn- DS\Agentic AI using Langgraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver


# ---------------------------
# State
# ---------------------------
class State(TypedDict):
    question: str
    answer: str


# ---------------------------
# Node
# ---------------------------
def chatbot(state: State):
    response = llm.invoke(state["question"])

    return {
        "answer": response
    }


# ---------------------------
# Build Graph
# ---------------------------
builder = StateGraph(State)

builder.add_node("chatbot", chatbot)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)


# ---------------------------
# Compile Graph
# ---------------------------
memory = MemorySaver()

workflow = builder.compile(
    checkpointer=memory
)


# ---------------------------
# Config
# ---------------------------
config = {
    "configurable": {
        "thread_id": "user123"
    }
}


# ---------------------------
# Streaming
# ---------------------------
for event in workflow.stream(
    {"question": "What is Artificial Intelligence?"},
    config=config,
    stream_mode="values"
):
    print(event)

{'question': 'What is Artificial Intelligence?'}
{'question': 'What is Artificial Intelligence?', 'answer': 'Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think, learn, and perform tasks that typically require human intelligence. This can include tasks such as visual perception, speech recognition, decision-making, and language translation.\n\nKey aspects of AI include:\n\n1. **Machine Learning (ML)**: A subset of AI that involves algorithms that can learn from and make predictions on data without being explicitly programmed. This is often used for tasks like image recognition, natural language processing, and recommendation systems.\n\n2. **Deep Learning**: A type of machine learning that uses neural networks with multiple layers to learn and extract features from raw data. It is particularly effective for complex tasks such as speech recognition, image and video analysis, and natural language processing.\n\n3. **Natural

In [4]:
response = llm.invoke("Hello")

print(response)
print(type(response))

Hello! How can I assist you today?
<class 'str'>
